# Parkinson's Disease Voice Screening Platform - End-to-End Training
## Complete Self-Contained Google Colab Training Pipeline (T4 GPU Free Tier)

> **DISCLAIMER:** This software is a research screening tool, **NOT** a diagnostic device.

### How to run this notebook in 3 steps:
1. **Select T4 GPU:** Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. **Run All Cells:** Click **Runtime** > **Run all** (or press `Ctrl+F9` / `Cmd+F9`).
3. **Download Weights:** The final cell will automatically download `best_model.pt` to your browser. You then copy `best_model.pt` to `models/artifact/best_model.pt` in your local project!

**Pipeline execution time:** ~4-5 minutes total on Colab T4 GPU.

In [ ]:
# Cell 1: Install dependencies and verify T4 GPU
!pip install -q transformers torch torchaudio soundfile huggingface_hub scikit-learn matplotlib tqdm

import torch
print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("WARNING: GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU")
    device = torch.device("cpu")


In [ ]:
# Cell 2: Download Italian Parkinson's Voice and Speech (IPVS) from Hugging Face
import os
from pathlib import Path
from huggingface_hub import snapshot_download

BASE_DIR = Path("/content/parkinsons_platform")
RAW_DIR = BASE_DIR / "data" / "raw" / "ipvs"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading IPVS dataset from Hugging Face mirror (birgermoell/Italian_Parkinsons_Voice_and_Speech)...")
snapshot_download(
    repo_id="birgermoell/Italian_Parkinsons_Voice_and_Speech",
    repo_type="dataset",
    local_dir=str(RAW_DIR),
    local_dir_use_symlinks=False,
    resume_download=True
)

wav_files = list(RAW_DIR.rglob("*.wav"))
print(f"Dataset downloaded successfully! Found {len(wav_files)} .wav audio recordings.")


In [ ]:
# Cell 3: Construct metadata and subject-level leakage-free split
import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

records = []
for wav_path in tqdm(sorted(list(RAW_DIR.rglob("*.wav"))), desc="Parsing metadata"):
    parts = wav_path.parts
    try:
        group_idx = next(i for i, p in enumerate(parts) if any(k in p for k in ["Young", "Elderly", "Parkinson"]))
    except StopIteration:
        continue

    group_str = parts[group_idx]
    fname = wav_path.name

    if "Young" in group_str:
        group = "young_healthy"
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"yhc_{sub_name}"
    elif "Elderly" in group_str:
        group = "elderly_healthy"
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"ehc_{sub_name}"
    elif "Parkinson" in group_str:
        group = "parkinsons"
        label = 1
        batch = parts[group_idx + 1].strip().replace(" ", "_")
        sub_name = parts[group_idx + 2].strip().replace(" ", "_")
        subject_id = f"pd_{batch}_{sub_name}"
    else:
        continue

    fname_upper = fname.upper()
    if fname_upper.startswith(("VA", "VE", "VI", "VO", "VU")):
        task_type = "vowel_sustain"
    elif fname_upper.startswith("PR"):
        task_type = "reading_passage"
    else:
        task_type = "syllable_repetition"

    try:
        info = sf.info(str(wav_path))
        duration = round(info.duration, 4)
        sr = info.samplerate
    except Exception:
        continue

    records.append({
        "file_name": fname,
        "raw_path": str(wav_path),
        "subject_id": subject_id,
        "group": group,
        "label": label,
        "task_type": task_type,
        "duration_sec": duration,
        "sample_rate": sr
    })

df = pd.DataFrame(records)

# Subject-level split
subject_df = df[["subject_id", "group", "label"]].drop_duplicates().reset_index(drop=True)
train_subs, temp_subs = train_test_split(
    subject_df, test_size=20, stratify=subject_df["group"], random_state=42
)
val_subs, test_subs = train_test_split(
    temp_subs, test_size=10, stratify=temp_subs["group"], random_state=42
)

sub_split_map = {}
for s in train_subs["subject_id"]: sub_split_map[s] = "train"
for s in val_subs["subject_id"]: sub_split_map[s] = "val"
for s in test_subs["subject_id"]: sub_split_map[s] = "test"

df["split"] = df["subject_id"].map(sub_split_map)
print("Split distribution (clips):")
print(df["split"].value_counts())


In [ ]:
# Cell 4: Audio Preprocessing (16kHz mono, silence trimming, peak normalization, 4.0s)
import numpy as np

def preprocess_audio(waveform: np.ndarray, orig_sr: int, target_sr: int = 16000, target_duration_sec: float = 4.0):
    # 1. Mono conversion
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=-1)
    waveform = waveform.astype(np.float32)
    
    # 2. Resample if necessary
    if orig_sr != target_sr:
        num_target_samples = int(round(len(waveform) * target_sr / orig_sr))
        waveform = np.interp(
            np.linspace(0, len(waveform), num_target_samples, endpoint=False),
            np.arange(len(waveform)),
            waveform
        ).astype(np.float32)
    
    # 3. Simple silence trimming (energy threshold)
    frame_len = int(0.025 * target_sr)
    hop_len = int(0.010 * target_sr)
    energy = np.array([
        np.sum(waveform[i:i + frame_len] ** 2)
        for i in range(0, max(1, len(waveform) - frame_len), hop_len)
    ])
    if len(energy) > 0 and np.max(energy) > 0:
        threshold = np.max(energy) * 0.001
        active_idx = np.where(energy >= threshold)[0]
        if len(active_idx) > 0:
            start = max(0, active_idx[0] * hop_len)
            end = min(len(waveform), (active_idx[-1] * hop_len) + frame_len)
            waveform = waveform[start:end]
            
    # 4. Peak normalization
    peak = np.max(np.abs(waveform))
    if peak > 1e-7:
        waveform = waveform / peak * 0.95
        
    # 5. Fixed length segment (target 64,000 samples = 4.0s)
    target_samples = int(target_sr * target_duration_sec)
    if len(waveform) < target_samples:
        repeats = int(np.ceil(target_samples / len(waveform)))
        waveform = np.tile(waveform, repeats)[:target_samples]
    else:
        waveform = waveform[:target_samples]
        
    return waveform

AUDIO_PROC_DIR = PROCESSED_DIR / "audio"
AUDIO_PROC_DIR.mkdir(parents=True, exist_ok=True)

processed_paths = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing audio"):
    data, sr = sf.read(row["raw_path"])
    proc_audio = preprocess_audio(data, sr)
    
    out_sub_dir = AUDIO_PROC_DIR / row["split"] / row["subject_id"]
    out_sub_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_sub_dir / f"{Path(row['raw_path']).stem}_clean.wav"
    sf.write(str(out_file), proc_audio, 16000)
    processed_paths.append(str(out_file))

df["processed_path"] = processed_paths
print("Audio preprocessing complete!")


In [ ]:
# Cell 5: Frozen WavLM-Base-Plus Feature Extraction (GPU accelerated)
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

FEATURE_DIR = PROCESSED_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("Loading frozen WavLM-Base-Plus model...")
wavlm_name = "microsoft/wavlm-base-plus"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(wavlm_name)
wavlm_model = WavLMModel.from_pretrained(wavlm_name).to(device)
wavlm_model.eval()
wavlm_model.requires_grad_(False)

feature_paths = []
BATCH_SIZE_FE = 16

for i in tqdm(range(0, len(df), BATCH_SIZE_FE), desc="Extracting WavLM embeddings"):
    batch_rows = df.iloc[i:i + BATCH_SIZE_FE]
    waveforms = []
    for _, r in batch_rows.iterrows():
        w, _ = sf.read(r["processed_path"])
        waveforms.append(w)
        
    inputs = feature_extractor(waveforms, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    
    with torch.inference_mode():
        outputs = wavlm_model(input_values)
        hidden_states = outputs.last_hidden_state.cpu().numpy()  # (batch, T=199, 768)
        
    for j, (_, r) in enumerate(batch_rows.iterrows()):
        feat = hidden_states[j]
        feat_sub_dir = FEATURE_DIR / r["split"] / r["subject_id"]
        feat_sub_dir.mkdir(parents=True, exist_ok=True)
        feat_file = feat_sub_dir / f"{Path(r['processed_path']).stem}_wavlm.npy"
        np.save(str(feat_file), feat)
        feature_paths.append(str(feat_file))

df["feature_path"] = feature_paths
df.to_csv(str(PROCESSED_DIR / "metadata.csv"), index=False)
print(f"Feature extraction complete! Cached {len(feature_paths)} feature matrices. Shape: {feat.shape}")


In [ ]:
# Cell 6: Define ParkinsonsVoiceClassifier Architecture
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.beta = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gx = torch.norm(x, p=2, dim=(1, 2), keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x * nx) + self.beta + x

class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.grn = GRN(4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        return residual + x

class ConvNeXtV2Stem(nn.Module):
    def __init__(self, in_features: int = 768, channels=(48, 96, 192), depths=(2, 2, 4)):
        super().__init__()
        self.initial_proj = nn.Sequential(
            nn.Conv2d(1, channels[0], kernel_size=(3, 7), stride=(1, 4), padding=(1, 3)),
            nn.GroupNorm(1, channels[0])
        )
        self.stages = nn.ModuleList()
        curr_dim = channels[0]
        for stage_idx in range(len(channels)):
            stage_blocks = []
            if stage_idx > 0:
                downsample = nn.Sequential(
                    nn.GroupNorm(1, curr_dim),
                    nn.Conv2d(curr_dim, channels[stage_idx], kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
                )
                stage_blocks.append(downsample)
                curr_dim = channels[stage_idx]
            for _ in range(depths[stage_idx]):
                stage_blocks.append(ConvNeXtV2Block(curr_dim))
            self.stages.append(nn.Sequential(*stage_blocks))
        self.out_channels = channels[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 3: x = x.unsqueeze(1)
        x = self.initial_proj(x)
        for stage in self.stages:
            x = stage(x)
        return x

class TransformerEncoderModule(nn.Module):
    def __init__(self, in_channels: int, freq_dim: int, d_model: int = 256, nhead: int = 4, num_layers: int = 3, num_tokens: int = 50, dropout: float = 0.3):
        super().__init__()
        self.token_proj = nn.Linear(in_channels * freq_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_tokens, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=1024, dropout=dropout, activation="gelu", batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, T_down, F_down = x.shape
        tokens = x.permute(0, 2, 1, 3).contiguous().view(B, T_down, C * F_down)
        tokens = self.token_proj(tokens)
        tokens = self.dropout(tokens + self.pos_embedding[:, :T_down, :])
        out = self.transformer(tokens)
        return self.layer_norm(out)

class AttentionPool(nn.Module):
    def __init__(self, d_model: int = 256):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.scale = d_model ** -0.5

    def forward(self, tokens: torch.Tensor):
        B = tokens.size(0)
        query = self.query.repeat(B, 1, 1)
        attn_logits = torch.bmm(query, tokens.transpose(1, 2)) * self.scale
        attn_weights = F.softmax(attn_logits, dim=-1)
        pooled = torch.bmm(attn_weights, tokens).squeeze(1)
        return pooled, attn_weights.squeeze(1)

class ParkinsonsVoiceClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = ConvNeXtV2Stem()
        self.encoder = TransformerEncoderModule(in_channels=192, freq_dim=48)
        self.pool = AttentionPool(d_model=256)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        tokens = self.encoder(x)
        pooled, attn_weights = self.pool(tokens)
        logits = self.head(pooled).squeeze(-1)
        return logits, attn_weights

model = ParkinsonsVoiceClassifier().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model instantiated! Total trainable parameters: {num_params:,}")


In [ ]:
# Cell 7: Dataset and DataLoaders
from torch.utils.data import Dataset, DataLoader

class VoiceFeatureDataset(Dataset):
    def __init__(self, metadata_df: pd.DataFrame):
        self.records = metadata_df.reset_index(drop=True)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        feat = np.load(row["feature_path"]).astype(np.float32)
        label = np.float32(row["label"])
        return torch.from_numpy(feat), torch.tensor(label)

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

n_neg = (train_df["label"] == 0).sum()
n_pos = (train_df["label"] == 1).sum()
pos_weight_val = n_neg / max(1, n_pos)
pos_weight = torch.tensor([pos_weight_val], device=device)
print(f"Train class counts: Neg (Healthy)={n_neg}, Pos (PD)={n_pos}. pos_weight={pos_weight_val:.4f}")

train_loader = DataLoader(VoiceFeatureDataset(train_df), batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VoiceFeatureDataset(val_df), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(VoiceFeatureDataset(test_df), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# Cell 8: Training Loop with Mixed Precision and Early Stopping
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

EPOCHS = 35
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.05 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

best_val_auc = 0.0
patience = 10
patience_counter = 0
best_model_path = BASE_DIR / "best_model.pt"

history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}

print("Starting model training on T4 GPU...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            logits, _ = model(feats)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * len(labels)
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    
    # Evaluation on validation set
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
                logits, _ = model(feats)
                loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    val_auc = roc_auc_score(all_labels, all_preds)
    val_acc = accuracy_score(all_labels, [1 if p >= 0.5 else 0 for p in all_preds])
    
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_auc"].append(val_auc)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val ROC-AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), str(best_model_path))
        print(f"  --> New best model saved! (Val AUC: {best_val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

print(f"Training finished! Best Validation ROC-AUC: {best_val_auc:.4f}")


In [ ]:
# Cell 9: Evaluate Best Model on Test Set & Download Weights
import matplotlib.pyplot as plt
from google.colab import files

# Load best model weights
model.load_state_dict(torch.load(str(best_model_path)))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for feats, labels in test_loader:
        feats = feats.to(device)
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            logits, _ = model(feats)
        probs = torch.sigmoid(logits).cpu().numpy()
        test_preds.extend(probs)
        test_labels.extend(labels.numpy())

test_auc = roc_auc_score(test_labels, test_preds)
test_acc = accuracy_score(test_labels, [1 if p >= 0.5 else 0 for p in test_preds])
test_f1 = f1_score(test_labels, [1 if p >= 0.5 else 0 for p in test_preds])

print("=" * 40)
print("FINAL HELD-OUT TEST SET EVALUATION")
print(f"Test ROC-AUC : {test_auc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")
print("=" * 40)

# Plot learning curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["val_auc"], label="Val ROC-AUC", color="green")
plt.plot(history["val_acc"], label="Val Accuracy", color="orange")
plt.title("Validation Metrics")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()

# Trigger browser download of best_model.pt
print("\nTriggering direct browser download of best_model.pt...")
files.download(str(best_model_path))
